# Sohn, Narain, Meirhaeghe & Jazayeri (2019) on DANDI:000130

Paper: Sohn H, Narain D, Meirhaeghe N, Jazayeri M (2019). Bayesian computation through cortical
latent dynamics. *Neuron* 103, 934-947. doi:10.1016/j.neuron.2019.06.012.

Dataset: [DANDI:000130](https://dandiarchive.org/dandiset/000130/0.241017.1448), version
0.241017.1448 (doi:10.48324/dandi.000130/0.241017.1448). This version was asked for in the
prompt of this attempt; it is also the dandiset's published version. The paper cites no version
of the dandiset, which was made later for the Neural Latents Benchmark.

Claims: `claims/sohn-2019.yaml`, written from the accepted author manuscript (PMC6805134):
Results, figure legends and STAR Methods; the supplementary tables and figures were not
available to the reader. This is a blind repeat attempt: it was made without the earlier
attempts on this dataset. The supplement (Tables S1 to S3) is under `paper/`, and Table S3 is
compared with the model fits in beh-bayes-model.

This is the revision after the first verification round: the production epoch now keeps the
two priors' ts = 800 trials apart, the Long prior's ts model of glm-prior-ts uses the Long
prior's windows, and the cells for the verifier's requested variants were added.

The dandiset holds one session (2016-12-11) of monkey H, one of the paper's 17 sessions of H and
29 of both animals. A "dataset" here, as in the paper, is one effector and target direction (4
here; the paper has 8, from two animals). The GPFA claims are in the second notebook,
`gpfa_sohn_2019.ipynb`.

| Claim | Tested here |
|---|---|
| beh-slopes, beh-overlap-prior, beh-sigmoid-increments, beh-long-more-bias, beh-bayes-model | yes |
| glm-prior-ts, dfr-overlap-support | yes |
| pca-dimensionality, encoding-r2, encoding-bayes-beats-linear, short-in-long-control, neural-long-more-bias | yes |
| speed-vs-projection, initial-speed-biased, speed-predicts-tp | yes |
| alt-speed-unmodulated, alt-transient-parallel, alt-threshold-monotonic | yes |
| gpfa-variance-inverted-u, gpfa-mean-sigmoid | yes, in `gpfa_sohn_2019.ipynb` |
| rnn-curved-manifold, rnn-perturbation | blocked: the networks are not in the dandiset (data_missing) |
| monkey-g | blocked: monkey G is not in the dandiset (data_missing) |

Method details that the claims file leaves open are taken from the paper, with the passage cited
in the cell or in `rsg.py`; every random generator is seeded per section.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import resource
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import rsg

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()
print("dandiset", rsg.DANDISET_ID, "version", rsg.VERSION)

## Data check

In [ ]:
sess = rsg.load_session()
tr = sess["trials"]
print(sess["meta"])
print("trials in the train file:", len(tr), "| by NLB split:", tr.split.value_counts().to_dict())
print("units:", len(sess["spikes"]), "| held out for NLB:", int(sess["heldout"].sum()))
print("observation intervals identical for all units:",
      all(np.array_equal(o, sess["obs_intervals"][0]) for o in sess["obs_intervals"]),
      "| observed", round(sess["duration"], 1), "s in", len(sess["obs_intervals"][0]), "intervals")
gaps = sess["obs_intervals"][0][1:, 0] - sess["obs_intervals"][0][:-1, 1]
print("gaps between observation intervals (s):", np.round(gaps, 3))
print("spike times sorted in every unit:", all(np.all(np.diff(s) >= 0) for s in sess["spikes"]))
print("spike time range (s):", round(min(s[0] for s in sess["spikes"]), 3), "to",
      round(max(s[-1] for s in sess["spikes"]), 3))

test, io = rsg.open_nwb(rsg.TEST_PATH)
tt = test.trials.to_dataframe()
print("test file: trials", len(tt), "columns", list(tt.columns), "units", len(test.units),
      "| segment length (s):", np.unique(np.round(tt.stop_time - tt.start_time, 3)))
io.close()

In [ ]:
print("completed (Set and Go present, no bad_time, tp > 0):", int(tr.completed.sum()))
print("completed and flagged is_outlier:", int((tr.completed & tr.is_outlier).sum()))
print("valid (completed, not outlier):", int(tr.valid.sum()))
print("trials with Go and a bad_time (left out):", int((tr.go_time.notna() & tr.bad_time.notna()).sum()))
v = tr[tr.valid].copy()
print("valid trials by split:", v.split.value_counts().to_dict())
print("ts values:", sorted(v.ts.unique()), "| iti by prior (ms):",
      v.groupby("prior").iti.unique().to_dict())
print("theta values:", sorted(tr.theta.unique()), "| is_eye:", sorted(tr.is_eye.unique()))
print("tp range (ms):", round(v.tp.min(), 1), "to", round(v.tp.max(), 1))
counts = v.groupby(["prior", "ts", "effector", "direction"]).size().unstack(["effector", "direction"])
print("valid trials per condition (min", int(counts.values.min()), ", max", int(counts.values.max()), "):")
print(counts)

# Event times against the interval columns
d_ts = (v.set_time - v.ready_time) * 1000 - v.ts
d_tp = (v.go_time - v.set_time) * 1000 - v.tp
print("\n(Set - Ready) - ts, ms: min", round(d_ts.min(), 1), "median", round(d_ts.median(), 1),
      "max", round(d_ts.max(), 1))
print("(Go - Set) - tp, ms: min", round(d_tp.min(), 1), "median", round(d_tp.median(), 1),
      "max", round(d_tp.max(), 1), "| |difference| > 60 ms in", int((d_tp.abs() > 60).sum()), "trials")
print("iti as the column description states (Short 500, Long 1220):",
      bool(((v.prior == "Short") == (v.iti == 500)).all()),
      "| as the Methods state (Short 1220, Long 500):", bool(((v.prior == "Short") == (v.iti == 1220)).all()))

unit_mask, unit_rate = rsg.select_units(sess)
print("\nunit mean rates (spikes/s): min", round(unit_rate.min(), 2), "max", round(unit_rate.max(), 2))
print("units below 1 spike/s:", list(sess["unit_ids"][unit_rate < 1]), np.round(unit_rate[unit_rate < 1], 2))
print("units selected:", int(unit_mask.sum()), "of", len(unit_mask))
spikes = [s for s, k in zip(sess["spikes"], unit_mask) if k]
print("run time so far", round(time.time() - T0), "s")

### Do the spikes share the clock of the trial table?

Target direction is randomised trial by trial (Methods, "Trial structure"), so it is a contrast
that an offset between the spike clock and the trial table would destroy. Direction is decoded
(5-fold cross-validated logistic regression, all 54 units) from spike counts in 200 ms windows
around Go, with the true event times and with the times shifted by 10 s, which moves the windows
into other trials. A standard choice of decoder; it is a check of the data, not a claim.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

rng = np.random.default_rng(0)
yv = (v.direction == "Left").values.astype(int)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
rows = []
for start in [-1000, -600, -200, 0, 200]:
    for shift in [0.0, 10.0]:
        c = rsg.bin_counts(sess["spikes"], v.go_time.values + shift, [start, start + 200])[:, :, 0]
        acc = cross_val_score(LogisticRegression(max_iter=2000), np.sqrt(c), yv, cv=cv).mean()
        rows.append(dict(window_from_go_ms=start, shift_s=shift, accuracy=round(acc, 3)))
clock = pd.DataFrame(rows).pivot(index="window_from_go_ms", columns="shift_s", values="accuracy")
print("direction decoding accuracy (chance about", round(max(yv.mean(), 1 - yv.mean()), 3), ")")
print(clock)

# Set-evoked population response: mean rate of all units around Set, true and shifted times
edges = np.arange(-400, 401, 20)
for shift in [0.0, 10.0]:
    c = rsg.bin_counts(sess["spikes"], v.set_time.values + shift, edges)
    pop = c.mean((0, 1)) / 0.02
    print(f"shift {shift:4.1f} s: population rate 200 ms before Set {pop[:20][-10:].mean():.2f}, "
          f"0-200 ms after {pop[20:30].mean():.2f} spikes/s; peak at {edges[1:][np.argmax(pop)]} ms")

## Behaviour

### beh-slopes

> Regression slopes of tp on ts are positive and less than one in every condition, so the animals
> estimated ts and their responses were biased toward the mean of the prior.

Criterion: per prior, effector and direction (8 conditions) a weighted regression of tp on ts,
each trial weighted by 1 over the variance of tp at its ts; all 8 slopes in (0, 1) with 95%
intervals excluding 0 and 1. The Methods write the regression as "tp = β ts + ε" (Methods,
"Model free analysis of behavior"). A slope below 1 can only show a bias toward the mean if the
line has an intercept, so the fit with an intercept is the main reading; the fit through the
origin is shown as the other reading.

In [ ]:
def slope_table(v):
    rows = []
    for (ds, prior), g in v.groupby(["dataset", "prior"]):
        w = 1 / g.groupby("ts").tp.transform("var")
        r = sm.WLS(g.tp, sm.add_constant(g.ts.astype(float)), weights=w).fit()
        r0 = sm.WLS(g.tp, g.ts.astype(float), weights=w).fit()
        lo, hi = r.conf_int().loc["ts"]
        lo0, hi0 = r0.conf_int().loc["ts"]
        rows.append(dict(dataset=ds, prior=prior, n=len(g), slope=r.params["ts"], ci_lo=lo, ci_hi=hi,
                         intercept=r.params["const"], slope_origin=r0.params["ts"],
                         ci_lo_origin=lo0, ci_hi_origin=hi0))
    return pd.DataFrame(rows)

slopes = slope_table(v)
print(slopes.round(3))
in_ci = ((slopes.ci_lo > 0) & (slopes.ci_hi < 1))
in_01 = ((slopes.slope > 0) & (slopes.slope < 1))
print("with intercept: slopes in (0,1):", int(in_01.sum()), "of 8; intervals excluding 0 and 1:", int(in_ci.sum()), "of 8")
in_01o = ((slopes.slope_origin > 0) & (slopes.slope_origin < 1))
in_cio = ((slopes.ci_lo_origin > 0) & (slopes.ci_hi_origin < 1))
print("through the origin: slopes in (0,1):", int(in_01o.sum()), "of 8; intervals excluding 0 and 1:", int(in_cio.sum()), "of 8")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2), sharey=True)
for ax, ds in zip(axes, rsg.DATASETS):
    g = v[v.dataset == ds]
    for prior, col in [("Short", "tab:red"), ("Long", "tab:blue")]:
        gp = g[g.prior == prior]
        ax.plot(gp.ts + rng.uniform(-10, 10, len(gp)), gp.tp, ".", ms=2, color=col, alpha=0.4)
        m = gp.groupby("ts").tp.mean()
        ax.plot(m.index, m.values, "o-", color=col)
    ax.plot([450, 1250], [450, 1250], "k--", lw=0.8)
    ax.set_title(ds); ax.set_xlabel("ts (ms)")
axes[0].set_ylabel("tp (ms)")
axes[0].set_ylim(300, 1700)
print("tp outside the axes (300 to 1700 ms):", int(((v.tp < 300) | (v.tp > 1700)).sum()), "trials")
print("plotted means of tp (ms) per dataset, prior and ts:")
print(v.groupby(["dataset", "prior", "ts"]).tp.mean().round(1).unstack("ts").to_string())
plt.tight_layout(); plt.show()

**Result: not reproduced.** With an intercept, 6 of 8 slopes lie between 0 and 1 (0.502 to
0.903), and both Eye Short slopes do not (1.011 and 1.107); 5 of 8 intervals exclude 0 and 1.
Through the origin, 2 of 8 slopes lie between 0 and 1. The slopes are clearly below 1 for the
Hand datasets (0.502 to 0.861) and near 1 for the Eye datasets (0.845 to 1.107).

### beh-overlap-prior

> At the overlapping ts of 800 ms, biases were in opposite directions for the two priors.

Criterion: ts = 800 trials pooled over effector and direction; median of tp - 800 negative for
Short and positive for Long, and a two-sided rank-sum test of tp between priors with p < 0.05.

In [ ]:
o = v[v.ts == 800]
rows = []
for name, g in [("pooled", o)] + list(o.groupby("dataset")):
    s, l = g[g.prior == "Short"].tp, g[g.prior == "Long"].tp
    p = stats.mannwhitneyu(s, l, alternative="two-sided").pvalue
    rows.append(dict(set=name, n_short=len(s), n_long=len(l), median_bias_short=np.median(s - 800),
                     median_bias_long=np.median(l - 800), mean_bias_short=np.mean(s - 800),
                     mean_bias_long=np.mean(l - 800), p_ranksum=p))
print(pd.DataFrame(rows).round(4))

**Result: not reproduced.** At ts = 800 ms the median bias is +27.8 ms for Short and +47.6 ms
for Long, both positive, and the rank-sum test gives p = 0.068 (92 Short, 107 Long trials). Long
tp is the longer of the two in 3 of 4 datasets by the mean; only Hand Right differs at p < 0.05
(p = 0.001), and in Hand Left the medians are -61.6 (Short) and -2.8 ms (Long).

### beh-sigmoid-increments

> Increments of tp between neighbouring ts are smaller at the extremes of the prior than near its
> mean.

Criterion: per condition (8), mean tp at each ts; extreme increment = mean of first and last,
middle = mean of second and third; middle exceeds extreme in most conditions, and a two-sided
signed-rank test over the 8 conditions has p < 0.05.

In [ ]:
rows = []
for (ds, prior), g in v.groupby(["dataset", "prior"]):
    m = g.groupby("ts").tp.mean().reindex(rsg.TS[prior]).values
    ext, mid = rsg.increments(m)
    rows.append(dict(dataset=ds, prior=prior, inc=np.round(np.diff(m), 1), extreme=ext, middle=mid))
inc = pd.DataFrame(rows)
print(inc.round(1))
print("middle > extreme in", int((inc.middle > inc.extreme).sum()), "of 8")
w = stats.wilcoxon(inc.middle, inc.extreme)
print("two-sided signed-rank: statistic", w.statistic, "p =", round(w.pvalue, 4),
      "| median middle - extreme", round(np.median(inc.middle - inc.extreme), 1), "ms")

**Result: reproduced.** The middle increment exceeds the extreme one in 7 of 8 conditions
(median difference 31.3 ms), two-sided signed-rank p = 0.0156; the exception is Hand Left Short.

### beh-long-more-bias

> Biases are larger for the Long prior than for the Short prior, as scalar variability predicts;
> regression slopes are smaller for Long.

Criterion: with the slopes of beh-slopes, Long below Short in all 4 datasets.

In [ ]:
cmp = slopes.pivot(index="dataset", columns="prior", values="slope")
cmp["long_below_short"] = cmp.Long < cmp.Short
cmp0 = slopes.pivot(index="dataset", columns="prior", values="slope_origin")
cmp["origin_long_below_short"] = cmp0.Long < cmp0.Short
print(cmp.round(3))
print("Long below Short (with intercept):", int(cmp.long_below_short.sum()), "of 4;",
      "through the origin:", int(cmp.origin_long_below_short.sum()), "of 4")

**Result: reproduced, on the assumption that the regression has an intercept.** The Long slope
is below the Short slope in 4 of 4 datasets (0.845 against 1.011, 0.903 against 1.107, 0.502
against 0.612, 0.612 against 0.861). Through the origin it is in 3 of 4, which would be partially
reproduced.

### beh-bayes-model

> A Bayesian observer model with scalar measurement and production noise fit to the behaviour
> captures the bias and the variance of tp in all conditions.

Criterion: the model fit per dataset (wm, wp, bShort, bLong); across the 10 prior and ts
conditions the R2 of the observed mean bias by the predicted bias (at least 0.8) and the Pearson
correlation of observed and predicted SD of tp (positive), in all 4 datasets. The R2 is computed
as 1 - SSres/SStot of the observed biases around the predicted ones; the squared correlation is
shown as the other reading.

In [ ]:
fits = {}
rows = []
for ds in rsg.DATASETS:
    g = v[v.dataset == ds]
    f = rsg.fit_bls(g)
    fits[ds] = f
    obs_b, pred_b, obs_sd, pred_sd = [], [], [], []
    for prior in rsg.PRIORS:
        lo, hi = rsg.TS[prior][0], rsg.TS[prior][-1]
        for ts in rsg.TS[prior]:
            x = g[(g.prior == prior) & (g.ts == ts)].tp
            _, m, sd = rsg.bls_predict(ts, f["wm"], f["wp"], f["b" + prior], lo, hi)
            obs_b.append(x.mean() - ts); pred_b.append(m - ts); obs_sd.append(x.std()); pred_sd.append(sd)
    obs_b, pred_b = np.array(obs_b), np.array(pred_b)
    r2 = 1 - np.sum((obs_b - pred_b) ** 2) / np.sum((obs_b - obs_b.mean()) ** 2)
    rows.append(dict(dataset=ds, wm=f["wm"], wp=f["wp"], bShort=f["bShort"], bLong=f["bLong"],
                     converged=f["success"], bias_R2=r2, bias_r2_corr=stats.pearsonr(obs_b, pred_b)[0] ** 2,
                     sd_r=stats.pearsonr(obs_sd, pred_sd)[0], sd_p=stats.pearsonr(obs_sd, pred_sd)[1],
                     obs_sd_range=f"{min(obs_sd):.0f}-{max(obs_sd):.0f}",
                     pred_sd_range=f"{min(pred_sd):.0f}-{max(pred_sd):.0f}"))
bayes = pd.DataFrame(rows)
print(bayes.round(3))
print("bias R2 >= 0.8 in", int((bayes.bias_R2 >= 0.8).sum()), "of 4; SD correlation > 0 in",
      int((bayes.sd_r > 0).sum()), "of 4")

Table S3 of the supplement gives, for monkey H, the mean ± SD over sessions of the parameters
fitted "to each session separately" (one fit per effector and direction). The fits of this
session are compared with them below, as the number of SDs from the mean.

In [ ]:
# Table S3 (supplement), monkey H: mean and SD over sessions of wm, wp, bShort and bLong (ms).
s3 = pd.DataFrame({"Hand Left": (0.085, 0.025, 0.086, 0.020, 0.394, 29.1, -42.0, 34.7),
                   "Hand Right": (0.062, 0.026, 0.084, 0.021, 9.25, 31.8, 5.77, 26.6),
                   "Eye Left": (0.043, 0.016, 0.086, 0.022, 23.4, 23.7, -4.16, 20.5),
                   "Eye Right": (0.027, 0.009, 0.077, 0.033, 0.847, 18.4, -10.8, 23.1)},
                  index=["wm", "wm_sd", "wp", "wp_sd", "bShort", "bShort_sd", "bLong", "bLong_sd"]).T
cmp_s3 = bayes.set_index("dataset")[["wm", "wp", "bShort", "bLong"]]
for par in ["wm", "wp", "bShort", "bLong"]:
    cmp_s3[par + "_S3"] = s3[par]
    cmp_s3[par + "_z"] = (cmp_s3[par] - s3[par]) / s3[par + "_sd"]
print(cmp_s3.round(3).to_string())
print("largest |z|:", round(cmp_s3[[c for c in cmp_s3 if c.endswith("_z")]].abs().max().max(), 2))

**Result: partially reproduced.** The predicted SD of tp correlates positively with the
observed SD in all 4 datasets (r = 0.28 to 0.91), but the bias R2 reaches 0.8 in 2 of 4: 0.978
(Hand Left) and 0.865 (Hand Right) against 0.671 (Eye Left) and 0.128 (Eye Right). The squared
correlation gives the same split (0.679, 0.135, 0.978, 0.868). In the Eye datasets the fitted
measurement noise is small (wm = 0.025 and 0.014, against 0.117 and 0.074 for Hand), so the model
predicts little bias. Every fitted parameter lies within 1.7 SD of the mean over sessions of
Table S3 (largest |z| 1.68); wm is below that mean in both Eye datasets (z = -1.11 and -1.46)
and above it in both Hand datasets (1.26 and 0.45).

## Single neurons

### glm-prior-ts

> About 30% of neurons were modulated by elapsed time (27% in monkey H) and more than 60% were
> sensitive to the prior (65% in monkey H), by a Poisson GLM of spike counts before Set compared
> by BIC.

Criterion: selected units; counts in the 80 ms before 480, 560, 640, 720 and 800 ms after Ready
from every trial whose Set comes at or after that time (attrition), both priors. Prior model:
log rate = 5 time indicators + prior + effector + direction, against the model without prior.
ts model per prior: that prior's 5 time indicators + effector + direction against intercept +
effector + direction; ts-dependent when BIC favours the full model in either prior. Expected
65 ± 6.5% and 27 ± 2.7%. For the Long prior the windows end at its ts values, 800 to 1200 ms
after Ready, from Long trials whose Set comes at or after the end of the window: the Methods use
"two distinct GLMs, one for data in the Short prior and one for the Long prior", because "values
of ts were different between the priors" (STAR Methods, "Generalized linear model"). The Short
prior's windows for the Long model are shown as variant (i).

The attrition rule includes the window that ends at Set itself (ts = t), which is the "80-ms
window immediately before Set" of the Methods ("Generalized linear model").

In [ ]:
def glm_design(sp, g, ends, attrition=True):
    """Counts in the 80 ms before each time in `ends` (ms after Ready), from the trials of g whose
    ts is at least that time (attrition) or, without attrition, at least the last time."""
    J, PRI, EFF, DIR, Y = [], [], [], [], []
    for j, t in enumerate(ends):
        gg = g[g.ts >= (t if attrition else ends[-1])]
        Y.append(rsg.bin_counts(sp, gg.ready_time.values, [t - 80, t])[:, :, 0])
        J.append(np.full(len(gg), j)); PRI.append((gg.prior == "Long").values.astype(float))
        EFF.append((gg.effector == "Hand").values.astype(float)); DIR.append((gg.direction == "Left").values.astype(float))
    J = np.concatenate(J)
    return dict(J=J, TI=(J[:, None] == np.arange(len(ends))[None, :]).astype(float), PRI=np.concatenate(PRI),
                EFF=np.concatenate(EFF), DIR=np.concatenate(DIR), Y=np.concatenate(Y))

def ts_bics(y, d):
    return (rsg.glm_bic(y, np.column_stack([d["TI"], d["EFF"], d["DIR"]])),
            rsg.glm_bic(y, np.column_stack([np.ones(len(y)), d["EFF"], d["DIR"]])))

def glm_units(sp, long_ends, attrition=True):
    both = glm_design(sp, v, rsg.TS["Short"], attrition)
    short = glm_design(sp, v[v.prior == "Short"], rsg.TS["Short"], attrition)
    long_ = glm_design(sp, v[v.prior == "Long"], long_ends, attrition)
    res = []
    for i in range(both["Y"].shape[1]):
        y = both["Y"][:, i]
        full = rsg.glm_bic(y, np.column_stack([both["TI"], both["PRI"], both["EFF"], both["DIR"]]))
        red = rsg.glm_bic(y, np.column_stack([both["TI"], both["EFF"], both["DIR"]]))
        bs, bl = ts_bics(short["Y"][:, i], short), ts_bics(long_["Y"][:, i], long_)
        fits = [full, red, *bs, *bl]
        res.append(dict(prior_dep=None not in (full, red) and full < red,
                        ts_dep_short=None not in bs and bs[0] < bs[1],
                        ts_dep_long=None not in bl and bl[0] < bl[1],
                        fails=sum(f is None for f in fits), mean_count=y.mean()))
    out = pd.DataFrame(res)
    out["ts_dep"] = out.ts_dep_short | out.ts_dep_long
    return out, both, long_

glm, both, long_d = glm_units(spikes, rsg.TS["Long"])
print("prior model observations:", len(both["J"]), "| per window ending 480..800 ms:", np.bincount(both["J"]),
      "| Long share:", round(both["PRI"].mean(), 3))
print("Long ts model observations per window ending 800..1200 ms:", np.bincount(long_d["J"]))
n = len(glm)
def binom_ci(k, n):
    ci = stats.binomtest(int(k), n).proportion_ci(0.95)
    return round(100 * float(ci.low), 1), round(100 * float(ci.high), 1)
kp, kt = int(glm.prior_dep.sum()), int(glm.ts_dep.sum())
print("units", n, "| failed fits", int(glm.fails.sum()))
print(f"prior-dependent: {kp}/{n} = {100*kp/n:.1f}%  95% CI {binom_ci(kp, n)}  (expected 65 +/- 6.5)")
print(f"ts-dependent:    {kt}/{n} = {100*kt/n:.1f}%  95% CI {binom_ci(kt, n)}  (expected 27 +/- 2.7)")
print("ts-dependent in the Short model:", int(glm.ts_dep_short.sum()), "| in the Long model:", int(glm.ts_dep_long.sum()))
print("both:", int((glm.prior_dep & glm.ts_dep).sum()), "| neither:", int((~glm.prior_dep & ~glm.ts_dep).sum()))

In [ ]:
# Variants: (i) the ts model of the Long prior on the Short prior's windows (480..800 ms);
# (ii) all 54 units; (iii) without attrition (each window from the trials whose ts is at least
# the last window's end).
def fracs(df):
    return round(100 * float(df.prior_dep.mean()), 1), round(100 * float(df.ts_dep.mean()), 1), int(df.fails.sum())
print("(i) Long ts model on 480..800 ms windows: prior %, ts %, failed fits =", fracs(glm_units(spikes, rsg.TS["Short"])[0]))
print("(ii) all 54 units: prior %, ts %, failed fits =", fracs(glm_units(sess["spikes"], rsg.TS["Long"])[0]))
print("(iii) no attrition: prior %, ts %, failed fits =", fracs(glm_units(spikes, rsg.TS["Long"], attrition=False)[0]))

**Result: not reproduced.** 38 of 48 units (79.2%, 95% interval 65.0 to 89.5%) are
prior-dependent and 22 of 48 (45.8%, 31.4 to 60.8%) are ts-dependent (16 in the Short model, 16
in the Long model), against 65 ± 6.5% and 27 ± 2.7%; no fit failed. Both fractions are above the
paper's, and the interval of the ts fraction excludes 27%. With the Short prior's windows for the
Long model the fractions are 79.2% and 50.0%; with all 54 units 75.9% and 44.4%; without
attrition 60.4% and 31.2%, which puts the prior fraction within tolerance and leaves the ts
fraction outside it.

### dfr-overlap-support

> At the overlapping ts of 800 ms, firing rates differ between priors, with the largest
> difference during the support of the prior, between 480 and 800 ms after Ready.

Criterion: per dataset, selected units, ts = 800 trials; per unit the difference of mean rates
between priors in 160 ms bins from 320 ms before Ready to 640 ms after Set; RMS over units,
averaged over the 4 datasets; the largest value in a bin within 480 to 800 ms after Ready and
above the 95th percentile of the largest value over 200 shuffles of the prior labels among
ts = 800 trials (within dataset).

In [ ]:
rng = np.random.default_rng(7)
edges = np.arange(-320, 800 + 640 + 1, 160)
centers = (edges[:-1] + edges[1:]) / 2
dat = {}
for ds in rsg.DATASETS:
    g = v[(v.dataset == ds) & (v.ts == 800)]
    c = rsg.bin_counts(spikes, g.ready_time.values, edges) / 0.16
    dat[ds] = (c, (g.prior == "Long").values)

def rms_curve(labels):
    out = []
    for ds in rsg.DATASETS:
        c, _ = dat[ds]
        L = labels[ds]
        d = c[L].mean(0) - c[~L].mean(0)
        out.append(np.sqrt((d ** 2).mean(0)))
    return np.array(out)

per_ds = rms_curve({ds: dat[ds][1] for ds in rsg.DATASETS})
curve = per_ds.mean(0)
null_max = []
for _ in range(200):
    null_max.append(rms_curve({ds: rng.permutation(dat[ds][1]) for ds in rsg.DATASETS}).mean(0).max())
thr = np.percentile(null_max, 95)
tab = pd.DataFrame(per_ds.T, index=[f"{a}..{b}" for a, b in zip(edges[:-1], edges[1:])], columns=rsg.DATASETS)
tab["mean"] = curve
print("RMS of the prior difference (spikes/s), bins in ms from Ready (Set at 800):")
print(tab.round(2))
print("trials per dataset (Short, Long):", {ds: (int((~dat[ds][1]).sum()), int(dat[ds][1].sum())) for ds in rsg.DATASETS})
kmax = int(np.argmax(curve))
print("largest value", round(curve[kmax], 2), "in bin", tab.index[kmax], "ms; shuffle 95th percentile of the maximum",
      round(thr, 2), "(median", round(np.median(null_max), 2), ")")
print("largest bin within 480-800 ms:", bool(edges[kmax] >= 480 and edges[kmax + 1] <= 800))
print("bins above the shuffle threshold:", list(tab.index[curve > thr]))

**Result: reproduced.** The mean RMS difference between priors at ts = 800 ms is largest in the
480 to 640 ms bin (6.88 spikes/s), above the 95th percentile of the shuffled maximum (3.13
spikes/s). Seven bins, from 160 to 1280 ms after Ready, exceed that threshold, so the difference
extends beyond the support of the prior; the two bins of the support are the 1st and 4th largest
(6.88 and 4.87 spikes/s).

## Population trajectories

Rates, PCA and the encoding axis follow the claims file's header (see `rsg.py`). Estimation
epoch: per prior, the trajectory from Ready to the prior's longest ts, each 20 ms bin averaging
the trials whose ts is at least the end of the bin. Production epoch: per ts, from Set to the
shortest tp of that ts. The state at time t is the bin ending at t.

In [ ]:
geo = {}
for ds in rsg.DATASETS:
    g = v[v.dataset == ds]
    est = {p: rsg.estimation_rates(spikes, g[g.prior == p], rsg.TS[p][-1]) for p in rsg.PRIORS}
    pe = rsg.pca([est[p][0] for p in rsg.PRIORS])
    prod = rsg.production_rates(spikes, g)
    pp = rsg.pca([prod[k][0] for k in sorted(prod)])
    geo[ds] = dict(g=g, est=est, pe=pe, prod=prod, pp=pp)
    print(ds, "| estimation bins", {p: len(est[p][2]) for p in rsg.PRIORS},
          "| production trajectories", len(prod))
    print("   production bins per (prior, ts)", {f"{k[0][0]}{k[1]}": len(prod[k][1]) for k in sorted(prod)})
    print("   shortest tp per (prior, ts)", {f"{k[0][0]}{k[1]}": round(prod[k][3]) for k in sorted(prod)})
r_all = np.concatenate([np.vstack([geo[ds]["est"][p][0] for p in rsg.PRIORS]) for ds in rsg.DATASETS])
print("smoothed rates (spikes/s), estimation epoch: min", round(r_all.min(), 2), "max", round(r_all.max(), 2))

### pca-dimensionality

> Population activity is low dimensional; 3 to 4 principal components in the estimation epoch
> and 5 to 10 in the production epoch explain about 75% of the variance.

Criterion: components needed for 75% per dataset and epoch; reproduced when all 8 fall in 3-4
(estimation) and 5-11 (production); partially when at least 4 do. Rates are not normalised
(claims file header). As variants, the counts after soft normalisation (rate / (range + 5
spikes/s), a common choice in this literature, not stated in the paper) and after z-scoring are
shown; they do not decide the outcome. The production epoch has 10 trajectories per dataset,
one per prior and ts.

In [ ]:
def k_for(blocks, frac=0.75, norm=None):
    X = np.vstack(blocks)
    if norm == "soft":
        X = X / (X.max(0) - X.min(0) + 5)
    elif norm == "z":
        X = (X - X.mean(0)) / (X.std(0) + 1e-9)
    return rsg.pca([X], frac)

rows = []
for ds in rsg.DATASETS:
    G = geo[ds]
    eb = [G["est"][p][0] for p in rsg.PRIORS]
    pb = [G["prod"][k][0] for k in sorted(G["prod"])]
    rows.append(dict(dataset=ds, k_est=G["pe"]["k"], k_prod=G["pp"]["k"],
                     cum_est_3=np.cumsum(G["pe"]["explained"])[2], cum_prod_5=np.cumsum(G["pp"]["explained"])[4],
                     k_est_soft=k_for(eb, norm="soft")["k"], k_prod_soft=k_for(pb, norm="soft")["k"],
                     k_est_z=k_for(eb, norm="z")["k"], k_prod_z=k_for(pb, norm="z")["k"]))
dims = pd.DataFrame(rows)
print(dims.round(3))
ok = ((dims.k_est >= 3) & (dims.k_est <= 4)).sum() + ((dims.k_prod >= 5) & (dims.k_prod <= 11)).sum()
print("values in range:", int(ok), "of 8")
for ds in rsg.DATASETS:
    print(ds, "estimation cumulative variance of PCs 1-5:", np.round(np.cumsum(geo[ds]["pe"]["explained"])[:5], 3),
          "| production:", np.round(np.cumsum(geo[ds]["pp"]["explained"])[:6], 3))

**Result: not reproduced.** 1 of 8 values falls in its range: 2 components reach 75% of the
variance in the estimation epoch of every dataset (3 to 4 expected) and 2, 2, 4 and 5 in the
production epoch (5 to 11 expected; Hand Right is in range). The first 3 components explain
0.868 to 0.907 of the estimation-epoch variance. As variants the claim does not use, z-scored
rates need 3 to 4 and 6 to 9 components (8 of 8 in range) and soft-normalised rates 3 and 4 to 7
(6 of 8).

### encoding-r2

> Projections of estimation-epoch neural states onto the encoding axis match the Bayesian
> estimates from the model fitted to the behaviour (R2 = 0.993 Short, 0.996 Long, monkey H, Hand
> Left).

Criterion: weighted regression Xu = a + b te over the 20 ms time points of the prior's support,
each point weighted by the number of trials it averages; te(t) is the model's mean estimate for
ts = t from the fit of beh-bayes-model to the same dataset. Readings: (a) Hand Left; (b) mean
over the 4 datasets. Reproduced when under one reading both R2 are within tolerance (at least
0.894 and 0.896). The R2 for 1000 random unit vectors in the same PC space is the reference.

In [ ]:
rng = np.random.default_rng(11)

def encoding_fit(X, n, ends, f, p, rng, n_rand=1000):
    """Encoding axis, Xu, te and the two weighted fits for one prior; R2 of random unit axes."""
    lo, hi = rsg.TS[p][0], rsg.TS[p][-1]
    u, nvec = rsg.encoding_axis(X, ends, lo, hi)
    sel = (ends >= lo) & (ends <= hi)
    t, xu, w = ends[sel], X[sel] @ u, n[sel]
    te = np.array([rsg.bls_predict(tt, f["wm"], f["wp"], f["b" + p], lo, hi)[0] for tt in t])
    fb, fl = rsg.wls(te, xu, w), rsg.wls(t.astype(float), xu, w)
    R = rng.standard_normal((n_rand, X.shape[1]))
    R /= np.linalg.norm(R, axis=1, keepdims=True)
    r2_rand = np.array([rsg.wls(te, X[sel] @ q, w)["r2"] for q in R])
    return dict(t=t, xu=xu, w=w, te=te, u=u, X=X, ends=ends, fb=fb, fl=fl, nvec=nvec, r2_rand=r2_rand)

enc = {}
rows = []
for ds in rsg.DATASETS:
    G, f = geo[ds], fits[ds]
    for p in rsg.PRIORS:
        r, n, ends = G["est"][p]
        e = encoding_fit(rsg.project(G["pe"], r), n, ends, f, p, rng)
        enc[(ds, p)] = e
        rows.append(dict(dataset=ds, prior=p, k=e["X"].shape[1], n_axis_vectors=e["nvec"], points=len(e["t"]),
                         weights=f"{e['w'].min()}-{e['w'].max()}", R2_bayes=e["fb"]["r2"],
                         R2_unweighted=stats.pearsonr(e["te"], e["xu"])[0] ** 2,
                         R2_random_median=np.median(e["r2_rand"]), R2_random_95=np.percentile(e["r2_rand"], 95),
                         frac_random_ge=np.mean(e["r2_rand"] >= e["fb"]["r2"])))
encr = pd.DataFrame(rows)
print(encr.round(4))
a = encr[encr.dataset == "Hand Left"].set_index("prior").R2_bayes
b = encr.groupby("prior").R2_bayes.mean()
print("(a) Hand Left: Short", round(a["Short"], 4), "Long", round(a["Long"], 4))
print("(b) mean of 4: Short", round(b["Short"], 4), "Long", round(b["Long"], 4))
print("within tolerance (Short >= 0.894, Long >= 0.896): (a)", bool(a["Short"] >= 0.894 and a["Long"] >= 0.896),
      "(b)", bool(b["Short"] >= 0.894 and b["Long"] >= 0.896))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharex=True)
for ax, ds in zip(axes, rsg.DATASETS):
    for p, col in [("Short", "tab:red"), ("Long", "tab:blue")]:
        e = enc[(ds, p)]
        fb = e["fb"]
        ax.plot(e["t"], (e["xu"] - fb["a"]) / fb["b"], "o", ms=3, color=col)
        ax.plot(e["t"], e["te"], "-", color=col)
    ax.plot([480, 1200], [480, 1200], "k--", lw=0.6)
    ax.set_title(ds); ax.set_xlabel("t after Ready (ms)")
axes[0].set_ylabel("Xu mapped to te (ms); line: te")
plt.tight_layout(); plt.show()
print("all points are inside the axes (autoscaled); plotted values, Xu mapped to te (ms) and te (ms):")
for (ds, p), e in enc.items():
    print(f"{ds:10s} {p:5s} t", e["t"][[0, -1]], "| mapped Xu", np.round((e["xu"] - e["fb"]["a"]) / e["fb"]["b"], 0).astype(int).tolist(),
          "| te", np.round(e["te"], 0).astype(int).tolist())

**Result: reproduced.** Under reading (a), Hand Left, R2 = 0.9957 (Short) and 0.9774 (Long);
under reading (b), the mean of 4 datasets, 0.9583 and 0.9823; both readings are within tolerance.
Random axes in the same 2-dimensional PC space give high R2 as well (median 0.716 to 0.987 per
dataset and prior), and the fraction of random axes at least as good as u ranges from 0.018 to
0.419, so the fit is not specific to u in this low-dimensional space. The normalised PC spaces
are shown after neural-long-more-bias.

### encoding-bayes-beats-linear

> Neural projections are better explained by the Bayesian model than by a linear function of ts
> with the same number of parameters, for both priors.

Criterion: RMSE of the weighted fits Xu = a + b te and Xu = a + b ts, per dataset and prior;
reproduced when the Bayesian RMSE is lower in all 4 datasets for both priors, partially when for
one prior. "RMSE" is read as the root mean squared residual of the weighted fit (Methods: "RMSE
between the actual and predicted Xu"); the weighted RMSE is shown as the other reading.

In [ ]:
rows = []
for (ds, p), e in enc.items():
    rows.append(dict(dataset=ds, prior=p, rmse_bayes=e["fb"]["rmse"], rmse_linear=e["fl"]["rmse"],
                     d_rmse=e["fb"]["rmse"] - e["fl"]["rmse"],
                     wrmse_bayes=e["fb"]["wrmse"], wrmse_linear=e["fl"]["wrmse"],
                     d_wrmse=e["fb"]["wrmse"] - e["fl"]["wrmse"]))
bl = pd.DataFrame(rows)
print(bl.round(4))
for p in rsg.PRIORS:
    s = bl[bl.prior == p]
    print(p, "| Bayesian lower in", int((s.d_rmse < 0).sum()), "of 4 (RMSE);", int((s.d_wrmse < 0).sum()),
          "of 4 (weighted RMSE); signed-rank p =", round(stats.wilcoxon(s.d_rmse).pvalue, 3))

**Result: partially reproduced.** The Bayesian RMSE is lower than the linear one in 4 of 4
datasets for Short and 3 of 4 for Long; in Hand Left Long it is higher (0.583 against 0.481). The
weighted RMSE gives the same counts.

### short-in-long-control

> In the Long prior during the period matched to the support of the Short prior, there is no
> sigmoidal representation; the Bayesian model is not better than the linear one.

Criterion (absence, on the paper's test): Long-prior states from 480 to 800 ms after Ready; an
axis from the state at 480 to that at 800 ms; compared with the Short prior's te and with ts as
above; reproduced when a two-sided signed-rank test of the RMSE difference over the 4 datasets
has p >= 0.05. With 4 datasets the smallest possible two-sided p is 0.125, so the test cannot find
the effect; the per-dataset differences are the effect size. The Methods say "the same projection
analysis", which would average the difference vectors as for u; that axis is shown too.

In [ ]:
def sil_fit(X, n, ends, f):
    """Short In Long: Long-prior states 480..800 ms, Short te; RMSE difference (Bayesian - linear)
    with the single axis (480 to 800) and with averaged difference vectors."""
    sel = (ends >= 480) & (ends <= 800)
    t, w = ends[sel], n[sel]
    te = np.array([rsg.bls_predict(tt, f["wm"], f["wp"], f["bShort"], 480, 800)[0] for tt in t])
    ends_l = list(ends)
    u1 = X[ends_l.index(800)] - X[ends_l.index(480)]; u1 /= np.linalg.norm(u1)
    u2, _ = rsg.encoding_axis(X, ends, 480, 800)
    row = dict(weights=f"{w.min()}-{w.max()}")
    for name, u in [("single", u1), ("averaged", u2)]:
        xu = X[sel] @ u
        fb, fl = rsg.wls(te, xu, w), rsg.wls(t.astype(float), xu, w)
        row[f"d_rmse_{name}"] = fb["rmse"] - fl["rmse"]
        row[f"r2_bayes_{name}"] = fb["r2"]
        row[f"r2_linear_{name}"] = fl["r2"]
    return row

rows = []
for ds in rsg.DATASETS:
    G, f = geo[ds], fits[ds]
    r, n, ends = G["est"]["Long"]
    row = dict(dataset=ds, **sil_fit(rsg.project(G["pe"], r), n, ends, f))
    s = enc[(ds, "Short")]
    row["d_rmse_short_prior"] = s["fb"]["rmse"] - s["fl"]["rmse"]
    rows.append(row)
sil = pd.DataFrame(rows)
print(sil.round(4))
for name in ["single", "averaged"]:
    print(name, "axis: signed-rank p =", round(stats.wilcoxon(sil[f"d_rmse_{name}"]).pvalue, 3),
          "| mean d_RMSE", round(sil[f"d_rmse_{name}"].mean(), 4))
print("Short prior mean d_RMSE for comparison:", round(sil.d_rmse_short_prior.mean(), 4))

**Result: reproduced.** By the paper's test the Bayesian model is not better than the linear one
in the Short In Long period: signed-rank p = 0.125 with the single axis and 0.25 with the averaged
axis. The test cannot reach p < 0.05 with 4 datasets. The Bayesian RMSE is still lower in 4 of 4
datasets with the single axis (mean difference -0.096, against -0.260 for the Short prior), so
the data show a smaller version of the Short-prior effect, not its absence.

### neural-long-more-bias

> Neural projections show more bias for the Long prior than for the Short prior, as a smaller
> regression slope of Xu on ts.

Criterion: slope of the weighted regression of Xu (spikes/s, in the dataset's shared PC space)
on ts (ms), per dataset and prior; Long below Short in all 4 datasets.

In [ ]:
rows = []
for ds in rsg.DATASETS:
    rows.append(dict(dataset=ds, slope_short=enc[(ds, "Short")]["fl"]["b"], slope_long=enc[(ds, "Long")]["fl"]["b"]))
nl = pd.DataFrame(rows)
nl["long_below_short"] = nl.slope_long < nl.slope_short
nl["ratio_long_short"] = nl.slope_long / nl.slope_short
print(nl.round(4))
print("Long below Short in", int(nl.long_below_short.sum()), "of 4; signed-rank p =",
      round(stats.wilcoxon(nl.slope_long - nl.slope_short).pvalue, 3))

**Result: partially reproduced.** The Long slope is below the Short slope in 3 of 4 datasets
(ratios 0.54, 0.62 and 0.36); in Hand Right they are nearly equal (0.0486 against 0.0470).

### The encoding claims in normalised PC spaces (variants)

The claims file fixes unnormalised rates, and in this session they give a 2-dimensional
estimation PC space in every dataset (pca-dimensionality). The paper does not say whether rates
were normalised. As variants that decide no outcome, the encoding claims are repeated in the PC
spaces of soft-normalised rates (rate / (range + 5 spikes/s)) and z-scored rates, each unit scaled
over the estimation-epoch matrix of both priors of its dataset, with the same 75% rule, the same
fits of the Bayesian model and 1000 random axes as the reference.

In [ ]:
def normalise(blocks, norm):
    X = np.vstack(blocks)
    if norm == "soft":
        scale = X.max(0) - X.min(0) + 5
        return [b / scale for b in blocks]
    mu, sd = X.mean(0), X.std(0) + 1e-9
    return [(b - mu) / sd for b in blocks]

rng = np.random.default_rng(12)
var_rows, var_sil = [], []
for norm in ["none", "soft", "z"]:
    for ds in rsg.DATASETS:
        G, f = geo[ds], fits[ds]
        blocks = [G["est"][p][0] for p in rsg.PRIORS]
        nb_ = blocks if norm == "none" else normalise(blocks, norm)
        pe = rsg.pca(nb_)
        for p, r in zip(rsg.PRIORS, nb_):
            _, n, ends = G["est"][p]
            e = enc[(ds, p)] if norm == "none" else encoding_fit(rsg.project(pe, r), n, ends, f, p, rng)
            var_rows.append(dict(norm=norm, dataset=ds, prior=p, k=e["X"].shape[1], R2_bayes=e["fb"]["r2"],
                                 R2_random_median=np.median(e["r2_rand"]), frac_random_ge=np.mean(e["r2_rand"] >= e["fb"]["r2"]),
                                 d_rmse=e["fb"]["rmse"] - e["fl"]["rmse"], slope_ts=e["fl"]["b"]))
        _, n, ends = G["est"]["Long"]
        var_sil.append(dict(norm=norm, dataset=ds, **sil_fit(rsg.project(pe, nb_[1]), n, ends, f)))
vr = pd.DataFrame(var_rows)
vs = pd.DataFrame(var_sil)
print(vr.round(4).to_string())
summ = []
for norm in ["none", "soft", "z"]:
    q = vr[vr.norm == norm]
    a = q[q.dataset == "Hand Left"].set_index("prior").R2_bayes
    b = q.groupby("prior").R2_bayes.mean()
    sl = q.pivot(index="dataset", columns="prior", values="slope_ts")
    s = vs[vs.norm == norm]
    summ.append(dict(norm=norm, k=f"{q.k.min()}-{q.k.max()}",
                     r2_HL_short=a["Short"], r2_HL_long=a["Long"], r2_mean_short=b["Short"], r2_mean_long=b["Long"],
                     encoding_r2_tolerance=bool((a["Short"] >= 0.894 and a["Long"] >= 0.896) or (b["Short"] >= 0.894 and b["Long"] >= 0.896)),
                     random_median=f"{q.R2_random_median.min():.3f}-{q.R2_random_median.max():.3f}",
                     frac_random_ge=f"{q.frac_random_ge.min():.3f}-{q.frac_random_ge.max():.3f}",
                     bayes_lower_short=int((q[q.prior == "Short"].d_rmse < 0).sum()),
                     bayes_lower_long=int((q[q.prior == "Long"].d_rmse < 0).sum()),
                     long_slope_below=int((sl.Long < sl.Short).sum()),
                     sil_p_single=stats.wilcoxon(s.d_rmse_single).pvalue, sil_bayes_lower_single=int((s.d_rmse_single < 0).sum())))
print(pd.DataFrame(summ).round(4).to_string())

**Variants, deciding no outcome.** In the soft-normalised space (3 components in every dataset)
and the z-scored space (3 to 4), encoding-r2 is within tolerance under both readings (Hand Left
0.9746 and 0.9809 soft, 0.9830 and 0.9693 z-scored), and random axes reach median R2 of 0.872 to
0.944 and 0.736 to 0.907. The Bayesian RMSE is lower in 4 and 4 of 4 datasets (soft) and 4 and 3
(z-scored); the Long slope is below the Short slope in 4 of 4 in both spaces. In Short In Long the
signed-rank p is 0.875 (soft) and 0.25 (z-scored), with the Bayesian RMSE lower in 2 and 1 of 4
datasets. So encoding-bayes-beats-linear and neural-long-more-bias would be reproduced in the
soft-normalised space and neural-long-more-bias in the z-scored one; the claims file fixes
unnormalised rates.

## Production epoch: speed

Speed is the mean Euclidean distance between successive 20 ms states in the production PC space,
from Set to the shortest tp of that ts (claims file header). The Methods' "divided by the
duration separating Set and the time of Go" is shown as a variant: that mean divided by the mean
tp of the ts.

### speed-vs-projection

> Larger projections on the encoding axis at Set are followed by slower neural trajectories in
> the production epoch (rho Short = -0.74, rho Long = -0.51).

Criterion: per prior, Xu at t = ts against the speed of that ts, pooled over the 4 datasets (20
points); reproduced when both correlations are within tolerance (-0.74 ± 0.074, -0.51 ±
0.051); partially when both are negative with p < 0.05 but one is outside, or when only one is
within tolerance.

In [ ]:
rows = []
for ds in rsg.DATASETS:
    G = geo[ds]
    for p in rsg.PRIORS:
        e = enc[(ds, p)]
        ends_l = list(e["ends"])
        for ts in rsg.TS[p]:
            rr, en, ntr, tmin = G["prod"][(p, ts)]
            Xp = rsg.project(G["pp"], rr)
            g = G["g"][(G["g"].prior == p) & (G["g"].ts == ts)]
            rows.append(dict(dataset=ds, prior=p, ts=ts, xu=e["X"][ends_l.index(ts)] @ e["u"],
                             speed=rsg.mean_step(Xp), speed_per_tp=rsg.mean_step(Xp) / g.tp.mean(),
                             mean_tp=g.tp.mean(), n=ntr, bins=len(en)))
spd = pd.DataFrame(rows)
print(spd.round(3).to_string())

def corr_rows(xcol, ycol):
    out = []
    for p in rsg.PRIORS:
        s = spd[spd.prior == p]
        r, pv = stats.pearsonr(s[xcol], s[ycol])
        out.append(dict(prior=p, set="pooled 20", r=r, p=pv))
        for ds in rsg.DATASETS:
            q = s[s.dataset == ds]
            r, pv = stats.pearsonr(q[xcol], q[ycol])
            out.append(dict(prior=p, set=ds, r=r, p=pv))
        z = s.groupby("dataset")[[xcol, ycol]].transform(lambda c: (c - c.mean()) / c.std())
        r, pv = stats.pearsonr(z[xcol], z[ycol])
        out.append(dict(prior=p, set="pooled, z-scored per dataset", r=r, p=pv))
    return pd.DataFrame(out)

print("\nXu at Set against speed")
print(corr_rows("xu", "speed").round(4))
print("\nvariant: speed divided by mean tp")
print(corr_rows("xu", "speed_per_tp").round(4))

**Result: partially reproduced.** Larger projections at Set go with slower trajectories in both
priors: pooled r = -0.931 (Short) and -0.874 (Long), both p < 1e-4, stronger than the paper's
-0.74 and -0.51 and outside the tolerance. Within datasets r ranges from -0.92 to -0.99, except
Hand Left Long (-0.67, p = 0.22). With speed divided by the mean tp, r = -0.916 and -0.925.

### initial-speed-biased

> The initial speed of production trajectories (Set to Set + 400 ms) is biased toward the middle
> of the prior; its dependence on ts is shallower than the unbiased profile proportional to 1/ts.

Criterion: per dataset and prior, the mean step distance between Set and Set + 400 ms (trials with
tp of at least 400 ms), projected in the production PC space; the slope of its regression on ts;
the slope of the unbiased profile speed(middle ts) x middle ts / ts. Reproduced when the measured
slope is shallower (less negative) than the unbiased one in most of the 8 and a two-sided
signed-rank test of the difference has p < 0.05.

In [ ]:
rows = []
for ds in rsg.DATASETS:
    G = geo[ds]
    for p in rsg.PRIORS:
        sp_ = []
        for ts in rsg.TS[p]:
            g = G["g"][(G["g"].prior == p) & (G["g"].ts == ts) & (G["g"].tp >= 400)]
            rr, _ = rsg.aligned_rates(spikes, g.set_time.values, 0, 400)
            sp_.append(rsg.mean_step(rsg.project(G["pp"], rr)))
        sp_ = np.array(sp_); tsv = np.array(rsg.TS[p], float)
        unb = sp_[2] * tsv[2] / tsv
        m_slope = np.polyfit(tsv, sp_, 1)[0]; u_slope = np.polyfit(tsv, unb, 1)[0]
        rows.append(dict(dataset=ds, prior=p, speeds=np.round(sp_, 3), slope_measured=m_slope,
                         slope_unbiased=u_slope, shallower=m_slope > u_slope))
ini = pd.DataFrame(rows)
print(ini.to_string())
d = ini.slope_measured - ini.slope_unbiased
print("measured shallower than unbiased in", int(ini.shallower.sum()), "of 8; signed-rank p =",
      round(stats.wilcoxon(d).pvalue, 4), "| median difference", f"{np.median(d):.2e}")

**Result: not reproduced.** The measured slope is shallower than the unbiased one in 7 of 8
dataset-priors (Eye Right Long is the exception), but the signed-rank test gives p = 0.0547,
above 0.05.

### speed-predicts-tp

> The speed of neural trajectories in the production epoch predicts the mean tp (rho Short =
> -0.58, rho Long = -0.52).

Criterion: per prior, the mean tp of each dataset and ts against the speed of that ts, pooled over
the 4 datasets (20 points); same rule as speed-vs-projection. Variant: trial-by-trial, within
each prior, effector and direction, the speed of each trial's smoothed rates (projected in the
production PC space, from Set to that trial's Go) against its tp, both z-scored within ts (the
Methods' trial-by-trial analysis, "Analysis of neural state variance", uses z-scoring within ts;
GPFA is not used for this variant).

In [ ]:
print("mean tp against speed")
print(corr_rows("mean_tp", "speed").round(4))
print("\nvariant: speed divided by mean tp")
print(corr_rows("mean_tp", "speed_per_tp").round(4))

rows = []
for ds in rsg.DATASETS:
    G = geo[ds]
    for p in rsg.PRIORS:
        g = G["g"][G["g"].prior == p]
        sps = []
        for _, tr_ in g.iterrows():
            stop = int(np.floor(tr_.tp / rsg.BIN_MS) * rsg.BIN_MS)
            c = rsg.bin_counts(spikes, [tr_.set_time], np.arange(0, stop + 1, rsg.BIN_MS))[0].T / 0.02
            sps.append(rsg.mean_step(rsg.project(G["pp"], rsg.smooth(c))))
        g = g.assign(speed=sps)
        z = g.groupby("ts")[["speed", "tp"]].transform(lambda c: (c - c.mean()) / c.std())
        r, pv = stats.pearsonr(z.speed, z.tp)
        rows.append(dict(dataset=ds, prior=p, n=len(g), r_trial=r, p=pv))
tt_ = pd.DataFrame(rows)
print("\nvariant: trial by trial, z-scored within ts")
print(tt_.round(4))

**Result: partially reproduced.** Pooled over datasets, r = -0.722 for Short and -0.744 for
Long (p = 0.0003 and 0.0002), both negative and both outside the tolerance (-0.58 ± 0.058 and
-0.52 ± 0.052). With speed divided by the mean tp, r = -0.861 and -0.860. Trial by trial,
z-scored within ts, r is negative in all 8 dataset-priors (-0.05 to -0.41) and p < 0.05 in 5 (the
4 Eye ones and Hand Right Short).

## Alternative mechanisms

### alt-speed-unmodulated

> Instantaneous speed of neural trajectories in the full neural space is not modulated during
> the support of each prior.

Criterion (absence, on the paper's test): per dataset and prior, the distance between successive
20 ms states of the smoothed rates of all selected units over the support (with attrition),
regressed on time; reproduced when a two-sided signed-rank test of the 4 slopes against zero has
p >= 0.05 for each prior. With 4 slopes the test cannot reach p < 0.05; the slopes and their
bootstrap intervals over trials (200 resamples of trials within each ts) are the effect size.

In [ ]:
rng = np.random.default_rng(21)

def support_slope(c, tsv, ends, lo, hi):
    use = tsv[:, None] >= ends[None, :]
    n = use.sum(0)
    r = rsg.smooth(np.einsum("tub,tb->bu", c, use.astype(float)) / np.maximum(n, 1)[:, None] / 0.02)
    sel = (ends >= lo) & (ends <= hi)
    steps = np.linalg.norm(np.diff(r[sel], axis=0), axis=1)
    tt = ends[sel][1:]
    return np.polyfit(tt, steps, 1)[0], steps

rows = []
for ds in rsg.DATASETS:
    g = v[v.dataset == ds]
    for p in rsg.PRIORS:
        gp = g[g.prior == p]
        lo, hi = rsg.TS[p][0], rsg.TS[p][-1]
        edges = np.arange(0, hi + rsg.BIN_MS, rsg.BIN_MS)
        c = rsg.bin_counts(spikes, gp.ready_time.values, edges)
        tsv = gp.ts.values
        slope, steps = support_slope(c, tsv, edges[1:], lo, hi)
        groups = [np.where(tsv == t)[0] for t in rsg.TS[p]]
        boots = []
        for _ in range(200):
            idx = np.concatenate([rng.choice(ix, len(ix)) for ix in groups])
            boots.append(support_slope(c[idx], tsv[idx], edges[1:], lo, hi)[0])
        rows.append(dict(dataset=ds, prior=p, slope_per_s=slope * 1000, ci_lo=np.percentile(boots, 2.5) * 1000,
                         ci_hi=np.percentile(boots, 97.5) * 1000, mean_step=steps.mean(),
                         step_range=f"{steps.min():.2f}-{steps.max():.2f}"))
asp = pd.DataFrame(rows)
print("slopes of step distance (spikes/s per 20 ms step) on time, per s:")
print(asp.round(3))
for p in rsg.PRIORS:
    s = asp[asp.prior == p]
    print(p, "signed-rank p =", round(stats.wilcoxon(s.slope_per_s).pvalue, 3), "| slopes > 0:",
          int((s.slope_per_s > 0).sum()), "of 4 | intervals excluding 0:", int(((s.ci_lo > 0) | (s.ci_hi < 0)).sum()),
          "| mean slope relative to mean step, per s:", round((s.slope_per_s / s.mean_step).mean(), 3))

The step distance between trial-averaged states has a noise floor that rises as fewer trials are
averaged, and with attrition the number of trials falls across the support. As a reference for
the slopes above, each trial's spike counts are permuted in time among the bins of the support
it contributes to (100 permutations per dataset and prior): this keeps every trial's counts and
the attrition, and removes any time course. The test is also run on unsmoothed rates, the
Methods' "full neural space to avoid any potential distortion by smoothing and PCA" (STAR
Methods, "Test of alternative mechanisms"). The bootstrap intervals above resample trials with
replacement, which adds noise to every state and lengthens every step, so they sit above the
point estimates; the permutation reference is the one to compare with.

In [ ]:
rng = np.random.default_rng(23)

def step_slope(c, tsv, ends, lo, hi, smooth=True):
    use = tsv[:, None] >= ends[None, :]
    r = np.einsum("tub,tb->bu", c, use.astype(float)) / np.maximum(use.sum(0), 1)[:, None] / 0.02
    if smooth:
        r = rsg.smooth(r)
    sel = (ends >= lo) & (ends <= hi)
    steps = np.linalg.norm(np.diff(r[sel], axis=0), axis=1)
    return np.polyfit(ends[sel][1:], steps, 1)[0] * 1000, steps.mean()

rows = []
for ds in rsg.DATASETS:
    g = v[v.dataset == ds]
    for p in rsg.PRIORS:
        gp = g[g.prior == p]
        lo, hi = rsg.TS[p][0], rsg.TS[p][-1]
        edges = np.arange(0, hi + rsg.BIN_MS, rsg.BIN_MS)
        ends = edges[1:]
        c = rsg.bin_counts(spikes, gp.ready_time.values, edges)
        tsv = gp.ts.values
        bins = [np.where((ends >= lo) & (ends <= t))[0] for t in tsv]
        for smooth in (True, False):
            obs, mstep = step_slope(c, tsv, ends, lo, hi, smooth)
            null = []
            for _ in range(100):
                c2 = c.copy()
                for i, b in enumerate(bins):
                    c2[i][:, b] = c[i][:, rng.permutation(b)]
                null.append(step_slope(c2, tsv, ends, lo, hi, smooth)[0])
            null = np.array(null)
            rows.append(dict(dataset=ds, prior=p, smoothed=smooth, slope_per_s=obs, null_mean=null.mean(),
                             null_lo=np.percentile(null, 2.5), null_hi=np.percentile(null, 97.5),
                             excess=obs - null.mean(), rel_excess=(obs - null.mean()) / mstep,
                             frac_null_ge=np.mean(null >= obs)))
aspn = pd.DataFrame(rows)
print(aspn.round(3).to_string())
for smooth in (True, False):
    for p in rsg.PRIORS:
        s = aspn[(aspn.smoothed == smooth) & (aspn.prior == p)]
        print("smoothed" if smooth else "unsmoothed", p, "| slopes signed-rank p =", round(stats.wilcoxon(s.slope_per_s).pvalue, 3),
              "| slopes > 0:", int((s.slope_per_s > 0).sum()), "of 4 | excess over the permutation mean > 0:", int((s.excess > 0).sum()),
              "of 4, signed-rank p =", round(stats.wilcoxon(s.excess).pvalue, 3),
              "| outside the permutation 95% range:", int(((s.slope_per_s < s.null_lo) | (s.slope_per_s > s.null_hi)).sum()), "of 4")

**Result: reproduced.** By the paper's test the slopes do not differ from zero: p = 0.25 (Short)
and 0.125 (Long); with 4 datasets the test cannot reach p < 0.05. The slopes are positive in 3 of
4 (Short) and 4 of 4 (Long), but with each trial's counts permuted in time they are as large or
larger: the observed slope exceeds the mean of the permutations in 1 of 4 (Short) and 0 of 4
(Long) datasets, and lies outside their 95% range in 2 and 1 of 4 (Eye Left Short above it,
Hand Left Short and Eye Left Long below it). On unsmoothed rates the slopes are 37 to 77 per s,
again near the permutations (above their mean in 1 of 4 and 2 of 4). The growth of the step
distance over the support is what the falling number of trials gives without any time course.

### alt-transient-parallel

> In the first 200 ms after Set, trajectories for different ts run parallel; their distance to
> the middle ts trajectory does not change.

Criterion (absence, on the paper's test): PCA per dataset on Set to Set + 200 ms, both priors, at
least 75% of variance; per dataset and prior the distance from the shortest and the longest ts
trajectory to the middle one at each 20 ms bin, regressed on time; reproduced when two-sided
signed-rank tests of the 8 slopes against zero have p >= 0.05 for both.

In [ ]:
rows = []
for ds in rsg.DATASETS:
    g = v[v.dataset == ds]
    tr_rates = {}
    for p in rsg.PRIORS:
        for ts in rsg.TS[p]:
            gg = g[(g.prior == p) & (g.ts == ts)]
            tr_rates[(p, ts)], ends = rsg.aligned_rates(spikes, gg.set_time.values, 0, 200)
    pt = rsg.pca(list(tr_rates.values()))
    for p in rsg.PRIORS:
        mid = rsg.project(pt, tr_rates[(p, rsg.TS[p][2])])
        for which, ts in [("shortest", rsg.TS[p][0]), ("longest", rsg.TS[p][-1])]:
            dist = np.linalg.norm(rsg.project(pt, tr_rates[(p, ts)]) - mid, axis=1)
            rows.append(dict(dataset=ds, prior=p, which=which, k=pt["k"], slope_per_s=np.polyfit(ends, dist, 1)[0] * 1000,
                             dist_first=dist[0], dist_last=dist[-1], dist_mean=dist.mean()))
atp = pd.DataFrame(rows)
print(atp.round(3))
for which in ["shortest", "longest"]:
    s = atp[atp.which == which]
    print(which, "| signed-rank p =", round(stats.wilcoxon(s.slope_per_s).pvalue, 4), "| slopes < 0:",
          int((s.slope_per_s < 0).sum()), "of 8 | median relative change over 200 ms:",
          round(np.median((s.dist_last - s.dist_first) / s.dist_mean), 3))

**Result: reproduced.** By the paper's test neither set of slopes differs from zero: p = 0.195
for the shortest ts and 1.0 for the longest. The distance of the shortest ts trajectory from the
middle one still grows in 6 of 8 dataset-priors (median +49% of its mean over 200 ms), while that
of the longest does not change (median -5%).

### alt-threshold-monotonic

> Aligned to the response, distances between trajectories for different ts decrease steadily up
> to movement initiation, without the converging then diverging pattern a ts-dependent threshold
> would produce.

Criterion: PCA per dataset on Go - 800 ms to Go, both priors, keeping 60% of the variance (Figure
5F legend); per dataset, prior and non-middle ts (32 curves) the distance to the middle ts
trajectory at each 20 ms bin and the time of its minimum. Reproduced when the median time of the
minimum is in the last 40 ms before Go; partially when in the last 200 ms. A bin's time is its end
(the last bin ends at Go). 75% of the variance is shown as a variant (the legend: "results remained
unchanged when more PCs were included").

In [ ]:
def threshold_minima(frac):
    rows = []
    for ds in rsg.DATASETS:
        g = v[v.dataset == ds]
        rr = {}
        for p in rsg.PRIORS:
            for ts in rsg.TS[p]:
                gg = g[(g.prior == p) & (g.ts == ts)]
                rr[(p, ts)], ends = rsg.aligned_rates(spikes, gg.go_time.values, -800, 0)
        pt = rsg.pca(list(rr.values()), frac)
        for p in rsg.PRIORS:
            mid = rsg.project(pt, rr[(p, rsg.TS[p][2])])
            for ts in rsg.TS[p]:
                if ts == rsg.TS[p][2]:
                    continue
                dist = np.linalg.norm(rsg.project(pt, rr[(p, ts)]) - mid, axis=1)
                rows.append(dict(dataset=ds, prior=p, ts=ts, k=pt["k"], t_min=ends[np.argmin(dist)],
                                 d_start=dist[0], d_min=dist.min(), d_end=dist[-1],
                                 monotonic_frac=np.mean(np.diff(dist) <= 0)))
    return pd.DataFrame(rows)

thr60 = threshold_minima(0.60)
print(thr60.round(2).to_string())
for name, df in [("60%", thr60), ("75%", threshold_minima(0.75))]:
    print(name, "| components", sorted(df.k.unique()), "| median time of minimum", df.t_min.median(), "ms",
          "| in last 40 ms:", int((df.t_min >= -20).sum()), "of 32 | in last 200 ms:", int((df.t_min >= -180).sum()),
          "| mean fraction of decreasing steps", round(df.monotonic_frac.mean(), 3))

**Result: not reproduced.** With 60% of the variance (1 to 3 components) the median time of the
minimum distance is 250 ms before Go; 4 of 32 minima are in the last 40 ms and 12 in the last 200
ms. With 75% (2 to 5 components) the median is 160 ms before Go, which would be partially
reproduced. About 57% of the 20 ms steps decrease the distance.

## Claims not tested

- rnn-curved-manifold, rnn-perturbation: the recurrent networks are the paper's simulations; no
  network, weights or code are in the dandiset or named as public by the paper. Blocked,
  data_missing.
- monkey-g: the dandiset holds one session of monkey H only. Blocked, data_missing.

In [ ]:
print("total run time", round(time.time() - T0), "s; peak memory",
      round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2), "GB")

## Summary

| Claim | Outcome |
|---|---|
| beh-slopes | not reproduced (2 Eye Short slopes above 1) |
| beh-overlap-prior | not reproduced (both biases positive, p = 0.068) |
| beh-sigmoid-increments | reproduced (7 of 8, p = 0.0156) |
| beh-long-more-bias | reproduced (4 of 4), assuming an intercept |
| beh-bayes-model | partially reproduced (bias R2 >= 0.8 in 2 of 4) |
| glm-prior-ts | not reproduced (79.2% and 45.8%) |
| dfr-overlap-support | reproduced |
| pca-dimensionality | not reproduced (1 of 8; 2 and 2 to 5 components) |
| encoding-r2 | reproduced (0.9957 and 0.9774, Hand Left) |
| encoding-bayes-beats-linear | partially reproduced (Short 4 of 4, Long 3 of 4) |
| short-in-long-control | reproduced (p = 0.125) |
| neural-long-more-bias | partially reproduced (3 of 4) |
| speed-vs-projection | partially reproduced (r = -0.93 and -0.87) |
| initial-speed-biased | not reproduced (7 of 8, p = 0.0547) |
| speed-predicts-tp | partially reproduced (r = -0.72 and -0.74) |
| alt-speed-unmodulated | reproduced (p = 0.25 and 0.125; slopes no larger than with time permuted) |
| alt-transient-parallel | reproduced (p = 0.195 and 1.0) |
| alt-threshold-monotonic | not reproduced (median minimum 250 ms before Go) |
| gpfa-variance-inverted-u, gpfa-mean-sigmoid | see `gpfa_sohn_2019.ipynb` |
| rnn-curved-manifold, rnn-perturbation, monkey-g | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- One session of monkey H (48 selected units, 982 valid trials, 4 datasets), where the paper uses
  17 sessions of H and 12 of G (8 datasets). The paper's PCA, GLM fractions and Bayesian fits pool
  sessions (Methods); here everything comes from one session, so the population has 48 units.
- Rates are not normalised and the kernel is truncated at the edges of each window (claims file
  and `rsg.py`).
- The behavioural regression is fitted with an intercept; the Methods write "tp = β ts + ε".
- The paper's speed model uses "the full neural space"; the claim's criterion uses smoothed
  rates, and unsmoothed rates are shown as a variant. Questions to a reader on beh-slopes,
  alt-speed-unmodulated and gpfa-variance-inverted-u are in `questions/sohn-2019.yaml`.

What was found about the dataset:

- The `iti` column's description gives 500 ms for Short and 1220 ms for Long; the data hold 1220
  ms for Short and 500 ms for Long, as the Methods state.
- `tp` is longer than Go - Set in every completed trial, by 0.1 to 153.3 ms (median 17.9 ms); the
  files do not say how `tp` was measured.
- The test file holds 283 NLB test segments of 1.7 s with no Ready, Set, ts, tp or condition, and
  40 of the 54 units; those trials cannot enter any analysis here.
- The dandiset holds one of the paper's 29 sessions; monkey G and the networks are not in it. The
  units carry no single- or multi-unit label and no electrode, and the electrodes have no
  positions.
- Event times and spikes share a clock: target direction is decoded at 0.62 to 0.75 from windows
  around Go and at chance (0.47 to 0.52) when the windows are shifted by 10 s.